# LLM Alpha Mining: 模拟大语言模型驱动的因子挖掘

## 论文参考

- **标题**: Automate Strategy Finding with LLM in Quant Investment
- **作者**: Zhizhuo Kou
- **年份**: 2024
- **关键结果**: 累计收益 53.17%

### 摘要

本文提出利用大语言模型 (LLM) 自动化量化投资策略挖掘流程。
LLM 根据提示 (Prompt) 生成 Alpha 因子公式，经回测验证后筛选最优因子。
本 notebook 模拟该流程: 定义模板库替代 LLM 输出，使用随机组合 + 爬山法搜索最佳因子。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### LLM Alpha Mining 流程

**1. Prompt 设计**

向 LLM 描述可用的数据字段和运算符，要求生成 Alpha 因子公式:

```
Available fields: close, open, high, low, volume
Available operators: rank, ts_mean, ts_std, delay, +, -, *, /
Generate 10 alpha factor formulas with high predictive power...
```

**2. 因子评估管线**

$$\text{LLM} \xrightarrow{\text{生成}} \text{公式} \xrightarrow{\text{求值}} \text{因子值} \xrightarrow{\text{评估}} \text{IC/IR} \xrightarrow{\text{筛选}} \text{最优因子}$$

**3. 评估指标**

- **IC**: $\text{corr}(\alpha_t, r_{t+1})$ - 因子预测能力
- **IR (Information Ratio)**: $\text{IC}_{\text{mean}} / \text{IC}_{\text{std}}$ - IC 的稳定性
- **Turnover**: 因子值变化率 - 交易成本代理

**4. 本实现的模拟策略**

由于不调用实际 LLM，我们用**模板因子库 + 参数搜索**模拟:
- 定义 20+ 个因子模板 (含参数槽位)
- 随机采样参数组合
- 爬山法 (Hill Climbing) 优化参数

In [ ]:
# ============================================================
# Cell 3: 动画展示 - LLM Alpha Mining 流程图
# ============================================================
import plotly.graph_objects as go

# 创建流程图动画: 逐步展示LLM管线的各个阶段
stages = [
    ('Prompt\nDesign', '设计提示词描述\n因子搜索空间', '#2196F3'),
    ('LLM\nGeneration', '(模拟) 生成\nAlpha因子公式', '#4CAF50'),
    ('Factor\nComputation', '在历史数据上\n计算因子值', '#FF9800'),
    ('IC/IR\nEvaluation', '评估预测能力\nIC, IR, Turnover', '#9C27B0'),
    ('Hill\nClimbing', '参数微调优化\n爬山搜索', '#F44336'),
    ('Best\nAlpha', '输出最佳\nAlpha因子', '#00BCD4'),
]

n_stages = len(stages)
x_pos = list(range(n_stages))

frames = []
for step in range(1, n_stages + 1):
    shapes = []
    annotations = []
    for i in range(step):
        name, desc, color = stages[i]
        # 方框
        shapes.append(dict(
            type='rect', x0=i-0.4, x1=i+0.4, y0=-0.5, y1=0.5,
            fillcolor=color, opacity=0.8, line=dict(color='white', width=2)
        ))
        annotations.append(dict(
            x=i, y=0.05, text=f'<b>{name}</b>', showarrow=False,
            font=dict(size=11, color='white')
        ))
        annotations.append(dict(
            x=i, y=-0.9, text=desc, showarrow=False,
            font=dict(size=9, color='#666')
        ))
        # 箭头
        if i > 0:
            annotations.append(dict(
                x=i-0.45, y=0, ax=i-0.55, ay=0,
                arrowhead=2, arrowsize=1.5, arrowwidth=2, arrowcolor='gray',
                showarrow=True, text=''
            ))

    frames.append(go.Frame(
        data=[go.Scatter(x=[], y=[], mode='markers')],  # dummy trace
        layout=go.Layout(shapes=shapes, annotations=annotations),
        name=f'Stage {step}'
    ))

fig = go.Figure(
    data=[go.Scatter(x=[], y=[])],
    frames=frames,
    layout=go.Layout(
        title=dict(text='LLM Alpha Mining Pipeline (模拟流程)'),
        xaxis=dict(visible=False, range=[-0.8, n_stages-0.2]),
        yaxis=dict(visible=False, range=[-1.5, 1]),
        height=350, width=950, plot_bgcolor='white',
        shapes=frames[0].layout.shapes,
        annotations=frames[0].layout.annotations,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 逐步展示', method='animate',
                 args=[None, {'frame': {'duration': 800}, 'transition': {'duration': 400}}]),
        ])],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import warnings
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)

sys.path.insert(0, '..')
from shared.data_fetcher import get_etf_daily, get_index_daily
from shared.backtest_engine import Backtester
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_metrics_table,
    plot_returns_distribution
)
from shared.constants import DEFAULT_START, DEFAULT_END, ETF_CSI300

print('所有模块导入成功')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================
try:
    df = get_etf_daily(ETF_CSI300, DEFAULT_START, DEFAULT_END)
    print(f'ETF 510300 数据: {len(df)} 条')
except Exception as e:
    print(f'数据获取异常: {e}, 使用模拟数据')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    n = len(dates)
    np.random.seed(42)
    price = 4.5 + np.cumsum(np.random.randn(n) * 0.02)
    price = np.maximum(price, 2.0)
    df = pd.DataFrame({
        'open': price * (1 + np.random.randn(n) * 0.003),
        'close': price,
        'high': price * (1 + np.abs(np.random.randn(n) * 0.008)),
        'low': price * (1 - np.abs(np.random.randn(n) * 0.008)),
        'volume': np.random.randint(50000, 2000000, n).astype(float),
    }, index=dates)

forward_ret = df['close'].pct_change(5).shift(-5)
print(f'数据范围: {df.index[0].strftime("%Y-%m-%d")} ~ {df.index[-1].strftime("%Y-%m-%d")}')

In [ ]:
# ============================================================
# Cell 6: 模板因子库 + IC/IR 评估
# ============================================================

def compute_rolling_ic(alpha_series, fwd_ret, window=60):
    """计算滚动IC序列"""
    combined = pd.DataFrame({'alpha': alpha_series, 'ret': fwd_ret}).dropna()
    if len(combined) < window:
        return pd.Series(dtype=float), 0, 0
    ic_series = combined['alpha'].rolling(window).corr(combined['ret'])
    ic_mean = ic_series.dropna().mean()
    ic_std = ic_series.dropna().std()
    ir = ic_mean / ic_std if ic_std > 0 else 0
    return ic_series, ic_mean, ir


def compute_turnover(alpha_series, window=1):
    """因子换手率: 因子排名日间变化"""
    ranked = alpha_series.rank(pct=True)
    return ranked.diff().abs().mean()


# 模板因子库 (模拟LLM生成)
ALPHA_TEMPLATES = {
    'momentum': lambda d, w: d['close'].pct_change(w),
    'mean_reversion': lambda d, w: d['close'].rolling(w).mean() / d['close'] - 1,
    'volume_surprise': lambda d, w: d['volume'] / d['volume'].rolling(w).mean() - 1,
    'price_range': lambda d, w: (d['high'] - d['low']).rolling(w).mean() / d['close'],
    'close_to_high': lambda d, w: (d['close'] - d['high'].rolling(w).max()) / d['close'],
    'close_to_low': lambda d, w: (d['close'] - d['low'].rolling(w).min()) / d['close'],
    'volume_momentum': lambda d, w: d['volume'].pct_change(w),
    'volatility': lambda d, w: d['close'].pct_change().rolling(w).std(),
    'vwap_deviation': lambda d, w: (d['close'] - (d['close'] * d['volume']).rolling(w).sum() / d['volume'].rolling(w).sum()) / d['close'],
    'open_close_gap': lambda d, w: ((d['close'] - d['open']) / d['open']).rolling(w).mean(),
    'high_low_momentum': lambda d, w: ((d['high'] - d['low']) / d['low']).pct_change(w),
    'volume_price_corr': lambda d, w: d['close'].rolling(w).corr(d['volume']),
    'rank_reversal': lambda d, w: -d['close'].pct_change(w).rank(pct=True),
    'gap_factor': lambda d, w: ((d['open'] - d['close'].shift(1)) / d['close'].shift(1)).rolling(w).mean(),
    'intraday_return': lambda d, w: ((d['close'] - d['open']) / d['open']).rolling(w).sum(),
    'overnight_return': lambda d, w: ((d['open'] - d['close'].shift(1)) / d['close'].shift(1)).rolling(w).sum(),
}

WINDOW_CHOICES = [3, 5, 10, 15, 20, 30]

print(f'模板因子库: {len(ALPHA_TEMPLATES)} 个模板')
print(f'窗口参数候选: {WINDOW_CHOICES}')
print(f'总搜索空间: {len(ALPHA_TEMPLATES) * len(WINDOW_CHOICES)} 个候选因子')

In [ ]:
# ============================================================
# Cell 7: 随机采样 + 爬山法搜索最佳因子
# ============================================================

# Phase 1: 全面扫描所有模板x窗口组合
print('Phase 1: 全面扫描所有因子组合...')
results = []

for name, func in ALPHA_TEMPLATES.items():
    for w in WINDOW_CHOICES:
        try:
            alpha = func(df, w)
            if alpha.isna().all() or alpha.std() == 0:
                continue
            ic_series, ic_mean, ir = compute_rolling_ic(alpha, forward_ret)
            turnover = compute_turnover(alpha)
            results.append({
                'name': name, 'window': w,
                'ic_mean': ic_mean, 'ir': ir,
                'abs_ic': abs(ic_mean), 'turnover': turnover,
                'alpha': alpha,
            })
        except Exception:
            continue

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'alpha'} for r in results])
results_df = results_df.sort_values('abs_ic', ascending=False)

print(f'\n成功评估 {len(results_df)} 个因子组合')
print('\nTop-10 因子 (按 |IC| 排序):')
print(results_df.head(10).to_string(index=False))

# Phase 2: 爬山法微调 Top-5 因子的参数
print('\n\nPhase 2: 爬山法优化 Top-5 因子...')
top5 = results_df.head(5)
hill_climb_results = []

for _, row in top5.iterrows():
    name = row['name']
    best_w = row['window']
    best_ic = row['abs_ic']
    func = ALPHA_TEMPLATES[name]

    # 在最佳窗口周围搜索
    for delta in [-2, -1, 1, 2, 3, 4, 5]:
        w_try = best_w + delta
        if w_try < 2: continue
        try:
            alpha = func(df, w_try)
            _, ic_mean, ir = compute_rolling_ic(alpha, forward_ret)
            if abs(ic_mean) > best_ic:
                best_ic = abs(ic_mean)
                best_w = w_try
        except:
            continue

    hill_climb_results.append({
        'name': name, 'optimized_window': best_w, 'optimized_ic': best_ic,
        'original_window': int(row['window']), 'original_ic': row['abs_ic'],
    })
    print(f'  {name}: window {int(row["window"])} -> {best_w}, IC {row["abs_ic"]:.4f} -> {best_ic:.4f}')

# 选择最佳因子
best_result = max(hill_climb_results, key=lambda x: x['optimized_ic'])
best_name = best_result['name']
best_window = best_result['optimized_window']
best_alpha = ALPHA_TEMPLATES[best_name](df, best_window)

print(f'\n最佳因子: {best_name}(window={best_window}), IC={best_result["optimized_ic"]:.4f}')

In [ ]:
# ============================================================
# Cell 8: 信号生成与回测
# ============================================================

# 将最佳Alpha因子转为交易信号
alpha_z = (best_alpha - best_alpha.rolling(60).mean()) / best_alpha.rolling(60).std()
alpha_z = alpha_z.dropna()

signals = pd.Series(0, index=alpha_z.index)
signals[alpha_z > 0.5] = 1
signals[alpha_z < -0.5] = -1

# 获取基准
try:
    bench = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
    bench_prices = bench['close']
except:
    bench_prices = None

backtester = Backtester(initial_capital=1_000_000, t_plus_1=True)
result = backtester.run(df['close'], signals, bench_prices)

print(f'=== LLM Alpha Mining (模拟) 回测结果 ===')
print(f'最佳因子: {best_name}(window={best_window})')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 收益曲线
benchmark_equity = None
if result.get('benchmark_returns') is not None:
    benchmark_equity = (1 + result['benchmark_returns']).cumprod() * result['equity_curve'].iloc[0]
plot_equity_curve(result['equity_curve'], benchmark_equity,
                  title=f'LLM Alpha ({best_name}) - 累计收益')

# 2. 回撤
plot_drawdown(result['equity_curve'], title='LLM Alpha - 回撤')

# 3. 因子IC分布
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# IC 分布条形图
top_results = results_df.head(15)
labels = [f"{r['name']}({int(r['window'])})" for _, r in top_results.iterrows()]
ics = top_results['abs_ic'].values
colors = ['#F44336' if ic > 0.03 else '#2196F3' for ic in ics]
ax1.barh(labels[::-1], ics[::-1], color=colors[::-1], alpha=0.8)
ax1.set_xlabel('|IC|')
ax1.set_title('Top-15 因子 IC 对比')
ax1.axvline(x=0.03, color='red', linestyle='--', alpha=0.5, label='IC=0.03')
ax1.legend()
ax1.grid(True, alpha=0.3)

# IR vs Turnover 散点图
ax2.scatter(results_df['turnover'], results_df['ir'],
            c=results_df['abs_ic'], cmap='RdYlGn', s=50, alpha=0.7)
ax2.set_xlabel('Turnover')
ax2.set_ylabel('IR')
ax2.set_title('因子 IR vs Turnover')
ax2.grid(True, alpha=0.3)
plt.colorbar(ax2.collections[0], ax=ax2, label='|IC|')

plt.tight_layout()
plt.show()

# 4. 绩效表
plot_metrics_table(result['metrics'], title='LLM Alpha Mining 绩效指标')

## 结果讨论

### 策略表现

1. **因子搜索效率**: 模板库 + 参数扫描在 ~100 个候选中快速定位高 IC 因子
2. **爬山法优化**: 在最佳因子附近微调窗口参数，进一步提升 IC
3. **IC/IR 权衡**: 高 IC 的因子不一定有高 IR，需综合考量因子稳定性

### 与实际 LLM 流程的差异

- 实际 LLM 可以生成更复杂的组合因子表达式
- LLM 具有跨域知识迁移能力，可融入宏观/事件信息
- 本模拟仅覆盖了规则化因子模板，搜索空间受限

### 改进方向

- 接入实际 LLM API (如 GPT-4) 生成因子公式
- 加入因子组合优化: 多因子加权/正交化
- 引入 IC 衰减分析评估因子时效性
- 多股票截面 IC 评估替代单资产回测